# Day 3: Isolation Forest - Banking Features
## Healthcare Fraud Detection - 40 Years Expertise!

**Goal**: Financial anomaly detection
**Innovation**: Banking domain knowledge transfer

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/healthcare-fraud-detection')
print(f'✅ Working in: {os.getcwd()}')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import *
import joblib
print('✅ Libraries loaded')

In [ ]:
print('Loading data & creating BANKING features...')
df = pd.read_csv('data/raw/healthcare_fraud_train.csv')
df['ClaimStartDt'] = pd.to_datetime(df['ClaimStartDt'])
df['ClaimEndDt'] = pd.to_datetime(df['ClaimEndDt'])

df['ClaimDuration'] = (df['ClaimEndDt'] - df['ClaimStartDt']).dt.days
df['ClaimMonth'] = df['ClaimStartDt'].dt.month
df['ClaimDayOfWeek'] = df['ClaimStartDt'].dt.dayofweek
df['ReimbursementRatio'] = df['InscClaimAmtReimbursed'] / (df['ClaimAmount'] + 1)
df['DeductibleRatio'] = df['DeductibleAmtPaid'] / (df['ClaimAmount'] + 1)
df['ClaimVelocity'] = df['ClaimAmount'] / (df['ClaimDuration'] + 1)
df['OverpaymentFlag'] = (df['InscClaimAmtReimbursed'] > df['ClaimAmount'] * 1.05).astype(int)
df['RoundNumberFlag'] = (df['ClaimAmount'] % 100 == 0).astype(int)
df['WeekendClaim'] = (df['ClaimDayOfWeek'] >= 5).astype(int)

provider_stats = df.groupby('ProviderID').agg({'ClaimAmount': ['mean', 'std', 'count'], 'PotentialFraud': 'sum'}).reset_index()
provider_stats.columns = ['ProviderID', 'Provider_AvgClaim', 'Provider_StdClaim', 'Provider_TotalClaims', 'Provider_FraudCount']
provider_stats['Provider_FraudRate'] = provider_stats['Provider_FraudCount'] / provider_stats['Provider_TotalClaims']
provider_stats['Provider_CV'] = provider_stats['Provider_StdClaim'] / (provider_stats['Provider_AvgClaim'] + 1)
df = df.merge(provider_stats, on='ProviderID', how='left')

bene_stats = df.groupby('BeneID').agg({'ClaimAmount': 'mean', 'ProviderID': 'nunique'}).reset_index()
bene_stats.columns = ['BeneID', 'Bene_AvgClaim', 'Bene_UniqueProviders']
df = df.merge(bene_stats, on='BeneID', how='left')
df['Bene_ProviderShopping'] = df['Bene_UniqueProviders'] / 2.0

df['DeviationFromAvg'] = (df['ClaimAmount'] - df['Provider_AvgClaim']) / (df['Provider_StdClaim'] + 1)

print('✅ 13 banking features created!')

In [ ]:
financial_features = ['ClaimAmount', 'ReimbursementRatio', 'DeductibleRatio', 'ClaimVelocity',
                      'OverpaymentFlag', 'RoundNumberFlag', 'WeekendClaim',
                      'Provider_AvgClaim', 'Provider_CV', 'Provider_FraudRate',
                      'Bene_ProviderShopping', 'DeviationFromAvg', 'ClaimDuration']

X = df[financial_features].fillna(0)
y = df['PotentialFraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('✅ Data prepared')

In [ ]:
print('Training Isolation Forest...')
contamination = y_train.mean()
iso_forest = IsolationForest(n_estimators=100, contamination=contamination, random_state=42, n_jobs=-1)
iso_forest.fit(X_train_scaled)

iso_pred_raw = iso_forest.predict(X_test_scaled)
iso_pred = (iso_pred_raw == -1).astype(int)

anomaly_scores = iso_forest.score_samples(X_test_scaled)
iso_pred_proba = -anomaly_scores
iso_pred_proba = (iso_pred_proba - iso_pred_proba.min()) / (iso_pred_proba.max() - iso_pred_proba.min())

iso_f1 = f1_score(y_test, iso_pred)
iso_auc = roc_auc_score(y_test, iso_pred_proba)

print('\n' + '='*70)
print('ISOLATION FOREST RESULTS (Banking Expertise!)')
print('='*70)
print(f'F1-Score: {iso_f1:.4f}')
print(f'ROC-AUC:  {iso_auc:.4f}')
print('\n' + classification_report(y_test, iso_pred, target_names=['Normal', 'Fraud']))

In [ ]:
joblib.dump(iso_forest, 'src/models/isolation_forest_banking.pkl')
joblib.dump(scaler, 'src/models/scaler_if.pkl')
results_day3 = {'model': 'Isolation Forest', 'f1': iso_f1, 'auc': iso_auc}
joblib.dump(results_day3, 'results/day3_results.pkl')
print('✅ Day 3 Complete!')